In [1]:
# ============================================================
# 18_AURORA_exposure_matched_lambda_reduced_universe_mechanism_figure.ipynb
# Single-cell standalone version
#
# Purpose:
# 1. Exposure-matched constant-lambda optimizer
# 2. Reduced-universe 0050/006208 sensitivity
# 3. Mechanism-attribution figure
#
# Main outputs:
# - table_S35_constant_lambda_exposure_match.csv
# - table_S36_reduced_universe_sensitivity.csv
# - table_S37_reduced_universe_exposure_diagnostics.csv
# - table_S38_constant_lambda_bootstrap.csv
# - table_S39_constant_lambda_grid_search.csv
# - figure_S4_mechanism_attribution.png
# - figure_S4_mechanism_attribution.pdf
# - all_notebook18_returns.parquet
# - all_notebook18_weights.parquet
# - all_notebook18_diagnostics.parquet
#
# Research backtest only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import copy
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
    print("scipy.optimize not available. Optimizer will use fallback allocations.")

# ============================================================
# 1. Paths and global settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"
NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "aurora_exposure_matched_lambda_reduced_universe" / f"run_{RUN_ID}"

RETURN_DIR = RUN_ROOT / "returns"
WEIGHT_DIR = RUN_ROOT / "weights"
TABLE_RUN_DIR = RUN_ROOT / "tables"
DIAG_DIR = RUN_ROOT / "diagnostics"
PLOT_DIR = RUN_ROOT / "plots"
REPORT_RUN_DIR = RUN_ROOT / "reports"

for d in [
    RUN_ROOT,
    RETURN_DIR,
    WEIGHT_DIR,
    TABLE_RUN_DIR,
    DIAG_DIR,
    PLOT_DIR,
    REPORT_RUN_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    REPORT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

BASE_ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
CLASS_LABELS = [0, 1, 2, 3, 4]

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0
TRANSACTION_COST_RATE = 0.0010
REBALANCE_FREQUENCY = "monthly"

STRICT_START = "2024-11-27"
STRICT_END = "2026-03-25"

BOOTSTRAP_REPLICATIONS = 5000
BOOTSTRAP_BLOCK_LENGTH = 20
BOOTSTRAP_RANDOM_SEED = 20260716

TARGET_AVG_CASH_DEFAULT = 0.5565
CASH_CAP_REPORTED = 0.60

CONSTANT_LAMBDA_GRID = np.unique(
    np.round(
        np.concatenate([
            np.linspace(2, 20, 19),
            np.linspace(22, 60, 20),
            np.array([10, 20, 30, 34, 40, 50, 55, 60, 70, 80]),
        ]),
        6,
    )
)

BASE_UAMV_B_CONFIG = {
    "strategy_name": "UAMV_B_more60_defensive",
    "alpha_20d": 0.30,
    "alpha_60d": 0.70,
    "lookback_mu": 63,
    "lookback_cov": 126,
    "mean_shrinkage_to_zero": 0.60,
    "momentum_weight": 0.40,
    "base_risk_aversion": 10.0,
    "uncertainty_risk_multiplier": 2.5,
    "bearish_risk_multiplier": 2.0,
    "turnover_penalty": 0.25,
    "regime_tilt_strength": 0.25,
    "max_etf_weight": 0.45,
    "max_00881_weight": 0.30,
    "max_cash_weight": CASH_CAP_REPORTED,
    "min_cash_weight": 0.00,
    "risk_aversion_mode": "dynamic",
    "constant_lambda": None,
    "constant_average_U": None,
    "constant_average_B": None,
}

print("=" * 90)
print("AURORA-TWETF Notebook 18")
print("Exposure-matched constant-lambda, reduced-universe sensitivity, and mechanism figure")
print("=" * 90)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("NOTEBOOK08_INPUT_INDEX:", NOTEBOOK08_INPUT_INDEX)
print("=" * 90)

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(f"Missing Notebook 07 probability input index: {NOTEBOOK08_INPUT_INDEX}")

# ============================================================
# 2. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
        .replace("-", "_")
        .replace("+", "plus")
        .replace("=", "_")
    )

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        df.index = pd.to_datetime(df.index)

    df.index.name = "date"
    return df.sort_index()

def load_base_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.sort_index()
    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in BASE_ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    df = df[BASE_ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df, path

def proba_cols():
    return [f"proba_class_{i}" for i in CLASS_LABELS]

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

def make_constant_probability_frame(index, prob_vec, model_name="constant_probability"):
    index = pd.DatetimeIndex(index).sort_values()
    prob_vec = np.asarray(prob_vec, dtype=float)
    prob_vec = prob_vec / prob_vec.sum()

    out = pd.DataFrame(index=index)
    out.index.name = "date"
    out["run_id"] = "NOTEBOOK18_CONSTANT"
    out["fold_id"] = "CONST"
    out["target_col"] = "constant_probability_target"
    out["split"] = "test"
    out["model_name"] = model_name
    out["model_family"] = "constant_probability_control"

    for i, p in enumerate(prob_vec):
        out[f"proba_class_{i}"] = float(p)

    return out

# ============================================================
# 3. Asset-aware allocation helper functions
# ============================================================

def all_assets_for(asset_list):
    return list(asset_list) + [CASH_COL]

def fallback_constrained_equal(asset_list):
    n = len(asset_list)
    if n <= 0:
        raise ValueError("asset_list must contain at least one risky asset.")
    return {asset: 1.0 / n for asset in asset_list} | {CASH_COL: 0.0}

def fallback_defensive(asset_list, cash_weight=0.20):
    n = len(asset_list)
    if n <= 0:
        raise ValueError("asset_list must contain at least one risky asset.")
    risky_each = (1.0 - cash_weight) / n
    return {asset: risky_each for asset in asset_list} | {CASH_COL: cash_weight}

def normalize_vector(w, asset_list):
    cols = all_assets_for(asset_list)
    w = pd.Series(w, index=cols, dtype=float)
    w = w.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    w[w < 0] = 0.0

    if w.sum() <= 0:
        w = pd.Series(fallback_constrained_equal(asset_list), dtype=float).reindex(cols).fillna(0.0)

    return w / w.sum()

def normalize_rows(df, asset_list):
    cols = all_assets_for(asset_list)
    out = df.copy()
    out = out.reindex(columns=cols).fillna(0.0)
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[out < 0] = 0.0

    row_sums = out.sum(axis=1)
    zero_mask = row_sums <= 0

    if zero_mask.any():
        out.loc[zero_mask, asset_list] = 1.0 / len(asset_list)
        out.loc[zero_mask, CASH_COL] = 0.0
        row_sums = out.sum(axis=1)

    out = out.div(row_sums, axis=0)
    return out

def cap_for_asset(asset, config):
    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    if asset == "00881":
        return min(max_etf_weight, max_00881_weight)
    return max_etf_weight

def apply_caps_to_vector(w, asset_list, config):
    cols = all_assets_for(asset_list)
    w = normalize_vector(w, asset_list)

    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])

    w[CASH_COL] = min(max(w[CASH_COL], min_cash_weight), max_cash_weight)

    for asset in asset_list:
        w[asset] = min(w[asset], cap_for_asset(asset, config))

    if w.sum() <= 0:
        w = pd.Series(fallback_constrained_equal(asset_list), dtype=float).reindex(cols).fillna(0.0)

    w = w / w.sum()

    for _ in range(20):
        excess = 0.0
        capped = []

        for asset in asset_list:
            cap = cap_for_asset(asset, config)
            if w[asset] > cap:
                excess += w[asset] - cap
                w[asset] = cap
                capped.append(asset)

        if w[CASH_COL] > max_cash_weight:
            excess += w[CASH_COL] - max_cash_weight
            w[CASH_COL] = max_cash_weight
            capped.append(CASH_COL)

        if w[CASH_COL] < min_cash_weight:
            needed = min_cash_weight - w[CASH_COL]
            w[CASH_COL] = min_cash_weight
            risky_sum = w[asset_list].sum()
            if risky_sum > 0:
                w[asset_list] *= max(0.0, risky_sum - needed) / risky_sum

        if excess <= 1e-12:
            break

        eligible = []
        for asset in cols:
            if asset in capped:
                continue
            if asset == CASH_COL:
                if w[asset] < max_cash_weight:
                    eligible.append(asset)
            else:
                if w[asset] < cap_for_asset(asset, config):
                    eligible.append(asset)

        if not eligible:
            break

        eligible_sum = w[eligible].sum()
        if eligible_sum <= 0:
            for asset in eligible:
                w[asset] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

        w = w.clip(lower=0.0)
        w = w / w.sum()

    return normalize_vector(w, asset_list)

def probability_features_from_blend(p20, p60, config):
    common = p20.index.intersection(p60.index).sort_values()

    p20 = p20.loc[common].copy()
    p60 = p60.loc[common].copy()

    prob20 = normalize_proba(p20[proba_cols()].values)
    prob60 = normalize_proba(p60[proba_cols()].values)

    alpha20 = float(config["alpha_20d"])
    alpha60 = float(config["alpha_60d"])

    if alpha20 + alpha60 <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        s = alpha20 + alpha60
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    prob = alpha20 * prob20 + alpha60 * prob60
    prob = normalize_proba(prob)

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = prob @ class_values

    clipped = np.clip(prob, 1e-12, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    confidence = 1.0 - normalized_entropy

    sorted_p = np.sort(prob, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (prob @ (class_values ** 2)) - expected_class ** 2

    out = pd.DataFrame(index=common)
    out.index.name = "date"
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_variance
    out["p_strong_bear"] = prob[:, 0]
    out["p_bear"] = prob[:, 1]
    out["p_neutral"] = prob[:, 2]
    out["p_bull"] = prob[:, 3]
    out["p_strong_bull"] = prob[:, 4]
    out["p_bearish"] = prob[:, 0] + prob[:, 1]
    out["p_bullish"] = prob[:, 3] + prob[:, 4]
    out["risk_on_score"] = (expected_class - 2.0) / 2.0

    for i, label in enumerate(CLASS_LABELS):
        out[f"proba_class_{label}"] = prob[:, i]

    return out

def fallback_weight(features_row, asset_list, config):
    p_bearish = float(features_row.get("p_bearish", 0.0))
    confidence = float(features_row.get("confidence_score", 0.5))

    if p_bearish > 0.45 and confidence > 0.25:
        base = pd.Series(fallback_defensive(asset_list, cash_weight=0.20), dtype=float)
    else:
        base = pd.Series(fallback_constrained_equal(asset_list), dtype=float)

    return apply_caps_to_vector(base.reindex(all_assets_for(asset_list)).fillna(0.0), asset_list, config)

def regime_tilt_for_assets(asset_list, features_row):
    risk_on = float(features_row.get("risk_on_score", 0.0))
    p_bearish = float(features_row.get("p_bearish", 0.0))
    p_bullish = float(features_row.get("p_bullish", 0.0))

    tilt = pd.Series(0.0, index=asset_list)

    for asset in asset_list:
        if asset in ["0050", "006208", "TW50_SYN"]:
            tilt[asset] += 0.15 * risk_on
        elif asset == "00692":
            tilt[asset] += 0.05 * risk_on + 0.10 * p_bearish
        elif asset == "00881":
            tilt[asset] += 0.35 * risk_on + 0.15 * p_bullish - 0.20 * p_bearish
        else:
            tilt[asset] += 0.10 * risk_on

    return tilt

def estimate_moments_for_date(etf_returns, date, features_row, config, asset_list):
    lookback_mu = int(config["lookback_mu"])
    lookback_cov = int(config["lookback_cov"])

    hist_all = etf_returns.loc[etf_returns.index < date, asset_list].copy()

    if len(hist_all) < max(30, min(lookback_mu, lookback_cov) // 2):
        return None, None

    hist_mu = hist_all.tail(lookback_mu)
    hist_cov = hist_all.tail(lookback_cov)

    if len(hist_mu) < 20 or len(hist_cov) < 30:
        return None, None

    mean_short = hist_mu.mean().values
    momentum_return = (1.0 + hist_mu).prod().values - 1.0
    momentum_daily = momentum_return / max(len(hist_mu), 1)

    momentum_weight = float(config["momentum_weight"])
    shrink = float(config["mean_shrinkage_to_zero"])

    mu_risky = (
        (1.0 - momentum_weight) * mean_short
        + momentum_weight * momentum_daily
    )
    mu_risky = (1.0 - shrink) * mu_risky

    tilt_strength = float(config["regime_tilt_strength"])
    tilt = regime_tilt_for_assets(asset_list, features_row)

    realized_vol = hist_cov.std().replace(0.0, np.nan)
    vol_scale = realized_vol.median()

    if not np.isfinite(vol_scale) or vol_scale <= 0:
        vol_scale = 0.01

    mu_risky = mu_risky + tilt_strength * tilt.values * vol_scale / ANNUALIZATION_DAYS

    cov_risky = hist_cov.cov().values
    cov_risky = np.nan_to_num(cov_risky, nan=0.0, posinf=0.0, neginf=0.0)

    avg_var = np.mean(np.diag(cov_risky))
    if not np.isfinite(avg_var) or avg_var <= 0:
        avg_var = 1e-4

    cov_risky = cov_risky + np.eye(len(asset_list)) * avg_var * 0.10

    cols = all_assets_for(asset_list)

    mu = np.zeros(len(cols), dtype=float)
    mu[:len(asset_list)] = mu_risky
    mu[-1] = 0.0

    cov = np.zeros((len(cols), len(cols)), dtype=float)
    cov[:len(asset_list), :len(asset_list)] = cov_risky
    cov[-1, -1] = 1e-10

    return mu, cov

def risk_aversion_for_date(features_row, config):
    mode = config.get("risk_aversion_mode", "dynamic")

    if mode == "constant_lambda":
        lam = float(config["constant_lambda"])
        return float(max(lam, 1e-6))

    base = float(config["base_risk_aversion"])
    uncertainty_mult = float(config["uncertainty_risk_multiplier"])
    bearish_mult = float(config["bearish_risk_multiplier"])

    if mode == "constant_average":
        uncertainty = float(config["constant_average_U"])
        p_bearish = float(config["constant_average_B"])
    else:
        uncertainty = float(features_row.get("normalized_entropy", 0.5))
        p_bearish = float(features_row.get("p_bearish", 0.0))

    lam = base * (1.0 + uncertainty_mult * uncertainty + bearish_mult * p_bearish)
    return float(max(lam, 1e-6))

def optimize_single_date(mu, cov, prev_w, features_row, config, asset_list):
    cols = all_assets_for(asset_list)

    if mu is None or cov is None or not HAS_SCIPY:
        return fallback_weight(features_row, asset_list, config), "fallback_no_moments_or_scipy"

    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])
    turnover_penalty = float(config["turnover_penalty"])

    lam = risk_aversion_for_date(features_row, config)
    prev = normalize_vector(prev_w, asset_list).values

    bounds = []
    for asset in cols:
        if asset == CASH_COL:
            bounds.append((min_cash_weight, max_cash_weight))
        else:
            bounds.append((0.0, cap_for_asset(asset, config)))

    def objective(w):
        w = np.asarray(w, dtype=float)
        expected_return = float(mu @ w)
        variance = float(w.T @ cov @ w)
        turnover_term = float(np.sum((w - prev) ** 2))
        utility = expected_return - lam * variance - turnover_penalty * turnover_term
        return -utility

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    x0 = apply_caps_to_vector(prev, asset_list, config).values

    try:
        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 300, "ftol": 1e-10, "disp": False},
        )

        if res.success and np.all(np.isfinite(res.x)):
            w = apply_caps_to_vector(res.x, asset_list, config)
            return w, "optimized"

        w = fallback_weight(features_row, asset_list, config)
        return w, f"fallback_optimizer_failed_{str(res.message)[:80]}"

    except Exception as e:
        w = fallback_weight(features_row, asset_list, config)
        return w, f"fallback_exception_{repr(e)[:80]}"

def build_uamv_signal_weights(p20, p60, etf_returns, config, evaluation_dates, asset_list):
    features = probability_features_from_blend(p20, p60, config)

    evaluation_dates = (
        pd.DatetimeIndex(evaluation_dates)
        .intersection(features.index)
        .intersection(etf_returns.index)
        .sort_values()
    )

    if len(evaluation_dates) == 0:
        raise ValueError(f"No evaluation dates for {config['strategy_name']}")

    cols = all_assets_for(asset_list)
    rows = []
    diagnostics = []

    prev_w = pd.Series(fallback_constrained_equal(asset_list), dtype=float).reindex(cols).fillna(0.0)
    prev_w = apply_caps_to_vector(prev_w, asset_list, config)

    for dt in evaluation_dates:
        features_row = features.loc[dt]

        mu, cov = estimate_moments_for_date(
            etf_returns=etf_returns,
            date=dt,
            features_row=features_row,
            config=config,
            asset_list=asset_list,
        )

        w, status = optimize_single_date(
            mu=mu,
            cov=cov,
            prev_w=prev_w,
            features_row=features_row,
            config=config,
            asset_list=asset_list,
        )

        rows.append(w.values)

        diag = {
            "date": dt,
            "strategy_name": config["strategy_name"],
            "optimization_status": status,
            "risk_aversion": risk_aversion_for_date(features_row, config),
            "expected_class": float(features_row["expected_class"]),
            "p_bearish": float(features_row["p_bearish"]),
            "p_bullish": float(features_row["p_bullish"]),
            "normalized_entropy": float(features_row["normalized_entropy"]),
            "confidence_score": float(features_row["confidence_score"]),
            "ordinal_variance": float(features_row["ordinal_variance"]),
            "risk_aversion_mode": config.get("risk_aversion_mode", "dynamic"),
            "constant_lambda": config.get("constant_lambda", np.nan),
            "asset_universe": ",".join(asset_list),
        }

        if mu is not None and cov is not None:
            diag["estimated_portfolio_mu"] = float(mu @ w.values)
            diag["estimated_portfolio_vol_daily"] = float(math.sqrt(max(w.values.T @ cov @ w.values, 0.0)))
        else:
            diag["estimated_portfolio_mu"] = np.nan
            diag["estimated_portfolio_vol_daily"] = np.nan

        diagnostics.append(diag)
        prev_w = w

    weight_df = pd.DataFrame(rows, index=evaluation_dates, columns=cols)
    weight_df.index.name = "date"
    weight_df = normalize_rows(weight_df, asset_list)

    diagnostic_df = pd.DataFrame(diagnostics).set_index("date").sort_index()
    diagnostic_df.index.name = "date"

    return weight_df, diagnostic_df, features.loc[evaluation_dates].copy()

# ============================================================
# 4. Backtest, metrics, diagnostics, bootstrap
# ============================================================

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    elif frequency == "daily":
        return idx
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    return pd.DatetimeIndex([values.iloc[0] for _, values in groups])

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates, asset_list):
    cols = all_assets_for(asset_list)
    signal = signal_weight_df.reindex(columns=cols).fillna(0.0).copy()
    signal = normalize_rows(signal, asset_list)
    signal_idx = pd.DatetimeIndex(signal.index).sort_values()

    daily = pd.DataFrame(index=daily_index, columns=cols, dtype=float)

    for i, reb_date in enumerate(rebalance_dates):
        if i + 1 < len(rebalance_dates):
            period_idx = daily_index[(daily_index >= reb_date) & (daily_index < rebalance_dates[i + 1])]
        else:
            period_idx = daily_index[daily_index >= reb_date]

        prior_signals = signal_idx[signal_idx < reb_date]
        if len(prior_signals) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        daily.loc[period_idx, cols] = signal.loc[signal_date, cols].values

    daily = daily.ffill().bfill()
    daily = normalize_rows(daily, asset_list)
    return daily

def compute_turnover(daily_weights, rebalance_dates, asset_list):
    cols = all_assets_for(asset_list)
    turnover = pd.Series(0.0, index=daily_weights.index)
    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt, cols]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_on_fixed_index(policy_name, signal_weights, etf_returns, evaluation_index, asset_list):
    cols = all_assets_for(asset_list)
    evaluation_index = pd.DatetimeIndex(evaluation_index).sort_values()

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0
    returns = returns.reindex(evaluation_index)

    if returns[cols].isna().any().any():
        missing_rows = returns[returns[cols].isna().any(axis=1)]
        raise ValueError(
            f"Return panel has missing rows for {policy_name}. "
            f"Example missing dates: {missing_rows.index[:5].tolist()}"
        )

    signal = signal_weights.copy()
    signal.index = pd.to_datetime(signal.index)
    signal = signal.sort_index()
    signal = signal.reindex(columns=cols).fillna(0.0)

    signal_aligned = signal.reindex(evaluation_index).ffill().bfill()
    signal_aligned = normalize_rows(signal_aligned, asset_list)

    rebalance_dates = get_rebalance_dates(evaluation_index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_aligned,
        daily_index=evaluation_index,
        rebalance_dates=rebalance_dates,
        asset_list=asset_list,
    )

    gross_return = (daily_weights[cols] * returns[cols]).sum(axis=1)
    turnover = compute_turnover(daily_weights, rebalance_dates, asset_list)
    transaction_cost = turnover * TRANSACTION_COST_RATE
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL
    drawdown = equity / equity.cummax() - 1.0

    out = pd.DataFrame(index=evaluation_index)
    out.index.name = "date"
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = drawdown
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def safe_index_label(x):
    if hasattr(x, "date"):
        return str(x.date())
    return str(x)

def performance_metrics_from_returns(r):
    r = pd.Series(r).dropna().astype(float)
    n = len(r)

    if n == 0:
        return {}

    equity = (1.0 + r).cumprod()
    drawdown = equity / equity.cummax() - 1.0

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(daily_vol) else np.nan
    sharpe = float((r.mean() / daily_vol) * np.sqrt(ANNUALIZATION_DAYS)) if daily_vol and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((r.mean() / downside_vol) * np.sqrt(ANNUALIZATION_DAYS)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = float(annual_return / abs(max_drawdown)) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": safe_index_label(r.index.min()),
        "end_date": safe_index_label(r.index.max()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "daily_volatility": daily_vol,
        "worst_daily_return": float(r.min()),
        "best_daily_return": float(r.max()),
        "final_equity": float(equity.iloc[-1]),
    }

def performance_metrics(return_df):
    return performance_metrics_from_returns(return_df["net_return"])

def exposure_diagnostics(weight_df, return_df, label, config=None, asset_list=None):
    if asset_list is None:
        asset_list = [c for c in weight_df.columns if c != CASH_COL]
    cols = all_assets_for(asset_list)

    w = weight_df.reindex(columns=cols).fillna(0.0)
    w = normalize_rows(w, asset_list)

    cash_cap = float(config["max_cash_weight"]) if config is not None else np.nan
    cap_tol = 1e-8

    risky = w[asset_list]
    max_asset_each_day = risky.max(axis=1)

    out = {
        "strategy_control": label,
        "asset_universe": ",".join(asset_list),
        "n_assets": int(len(asset_list)),
        "avg_cash_weight": float(w[CASH_COL].mean()),
        "median_cash_weight": float(w[CASH_COL].median()),
        "min_cash_weight": float(w[CASH_COL].min()),
        "max_cash_weight_realized": float(w[CASH_COL].max()),
        "pct_days_at_cash_cap": (
            float((w[CASH_COL] >= cash_cap - cap_tol).mean())
            if np.isfinite(cash_cap)
            else np.nan
        ),
        "avg_equity_exposure": float(risky.sum(axis=1).mean()),
        "median_equity_exposure": float(risky.sum(axis=1).median()),
        "avg_max_risky_asset_weight": float(max_asset_each_day.mean()),
        "median_max_risky_asset_weight": float(max_asset_each_day.median()),
        "total_turnover": float(return_df["turnover"].sum()) if "turnover" in return_df else np.nan,
        "avg_turnover": float(return_df["turnover"].mean()) if "turnover" in return_df else np.nan,
        "total_transaction_cost": float(return_df["transaction_cost"].sum()) if "transaction_cost" in return_df else np.nan,
        "num_rebalance_days": int(return_df["is_rebalance_date"].sum()) if "is_rebalance_date" in return_df else np.nan,
    }

    for asset in asset_list:
        out[f"avg_{asset}_weight"] = float(w[asset].mean())

    if "00881" not in asset_list:
        out["avg_00881_weight"] = np.nan

    return out

def condition_number_stats(etf_returns, evaluation_index, asset_list, lookback=126):
    vals = []
    for dt in pd.DatetimeIndex(evaluation_index).sort_values():
        hist = etf_returns.loc[etf_returns.index < dt, asset_list].tail(lookback)
        if len(hist) < max(30, lookback // 2):
            continue
        cov = hist.cov().values
        cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
        avg_var = np.mean(np.diag(cov))
        if not np.isfinite(avg_var) or avg_var <= 0:
            continue
        cov = cov + np.eye(len(asset_list)) * avg_var * 0.10
        try:
            vals.append(float(np.linalg.cond(cov)))
        except Exception:
            continue

    if not vals:
        return {
            "median_cov_condition_number": np.nan,
            "p90_cov_condition_number": np.nan,
            "max_cov_condition_number": np.nan,
        }

    arr = np.asarray(vals, dtype=float)
    return {
        "median_cov_condition_number": float(np.nanmedian(arr)),
        "p90_cov_condition_number": float(np.nanpercentile(arr, 90)),
        "max_cov_condition_number": float(np.nanmax(arr)),
    }

def circular_block_indices(n, block_length, rng):
    idx = []
    while len(idx) < n:
        start = int(rng.integers(0, n))
        block = [(start + j) % n for j in range(block_length)]
        idx.extend(block)
    return np.asarray(idx[:n], dtype=int)

def paired_difference_metrics(strategy_returns, comparator_returns):
    s = pd.Series(strategy_returns).dropna().astype(float)
    c = pd.Series(comparator_returns).dropna().astype(float)

    common = s.index.intersection(c.index).sort_values()
    if len(common) > 0:
        s = s.loc[common]
        c = c.loc[common]
    else:
        if len(s) != len(c):
            raise ValueError("No common index and unequal return lengths.")
        s = s.reset_index(drop=True)
        c = c.reset_index(drop=True)

    ms = performance_metrics_from_returns(s)
    mc = performance_metrics_from_returns(c)

    return {
        "n_days": int(len(s)),
        "diff_total_return": ms["total_return"] - mc["total_return"],
        "diff_sharpe": ms["sharpe_ratio"] - mc["sharpe_ratio"],
        "diff_sortino": ms["sortino_ratio"] - mc["sortino_ratio"],
        "drawdown_improvement": ms["max_drawdown"] - mc["max_drawdown"],
        "strategy_total_return": ms["total_return"],
        "comparator_total_return": mc["total_return"],
        "strategy_sharpe": ms["sharpe_ratio"],
        "comparator_sharpe": mc["sharpe_ratio"],
        "strategy_sortino": ms["sortino_ratio"],
        "comparator_sortino": mc["sortino_ratio"],
        "strategy_max_drawdown": ms["max_drawdown"],
        "comparator_max_drawdown": mc["max_drawdown"],
    }

def paired_circular_block_bootstrap(strategy_returns, comparator_returns, n_rep=5000, block_length=20, seed=42):
    s = pd.Series(strategy_returns).dropna().astype(float)
    c = pd.Series(comparator_returns).dropna().astype(float)

    common = s.index.intersection(c.index).sort_values()
    if len(common) > 0:
        s = s.loc[common]
        c = c.loc[common]
    else:
        if len(s) != len(c):
            raise ValueError("No common index and unequal return lengths.")
        s = s.reset_index(drop=True)
        c = c.reset_index(drop=True)

    n = len(s)
    if n <= block_length:
        raise ValueError(f"Too few observations for block bootstrap: n={n}, block_length={block_length}")

    observed = paired_difference_metrics(s, c)

    rng = np.random.default_rng(seed)

    metrics = [
        "diff_total_return",
        "diff_sharpe",
        "diff_sortino",
        "drawdown_improvement",
    ]

    dist = {m: [] for m in metrics}
    s_values = s.values
    c_values = c.values

    for _ in range(n_rep):
        idx = circular_block_indices(n=n, block_length=block_length, rng=rng)

        bs = pd.Series(s_values[idx]).reset_index(drop=True)
        bc = pd.Series(c_values[idx]).reset_index(drop=True)

        bdiff = paired_difference_metrics(bs, bc)

        for m in metrics:
            dist[m].append(bdiff[m])

    rows = []

    for m in metrics:
        arr = np.asarray(dist[m], dtype=float)
        arr = arr[np.isfinite(arr)]

        ci_low, ci_high = np.percentile(arr, [2.5, 97.5])
        obs = float(observed[m])

        if ci_low > 0:
            result = "Significant positive"
        elif ci_high < 0:
            result = "Significant negative"
        else:
            result = "Not significant"

        rows.append({
            "metric": m,
            "observed_difference": obs,
            "ci95_lower": float(ci_low),
            "ci95_upper": float(ci_high),
            "result": result,
            "bootstrap_replications": int(n_rep),
            "block_length": int(block_length),
            "ci_type": "percentile",
        })

    return pd.DataFrame(rows)

# ============================================================
# 5. Load data and probability inputs
# ============================================================

print("\n" + "=" * 90)
print("Loading ETF returns and Notebook 07 probability inputs")
print("=" * 90)

base_etf_returns, etf_return_path = load_base_etf_return_panel()

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

p20_path = None
p60_path = None

for _, row in input_index_df.iterrows():
    target_col = row["target_col"]
    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col:
        p20_path = path
    elif "60d" in target_col:
        p60_path = path

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files.")

p20_raw = read_table_auto(p20_path)
p60_raw = read_table_auto(p60_path)

for name, prob_df in [("p20_raw", p20_raw), ("p60_raw", p60_raw)]:
    missing = [c for c in proba_cols() if c not in prob_df.columns]
    if missing:
        raise ValueError(f"{name} missing probability columns: {missing}")
    if "fold_id" not in prob_df.columns:
        raise ValueError(f"{name} must include fold_id.")
    if "split" not in prob_df.columns:
        raise ValueError(f"{name} must include split.")

p20_test_latest = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test_latest = latest_fold_deduplicate(p60_raw, split_filter=["test"])

aligned_test_dates = (
    p20_test_latest.index
    .intersection(p60_test_latest.index)
    .intersection(base_etf_returns.index)
    .sort_values()
)

aligned_test_dates = aligned_test_dates[
    (aligned_test_dates >= pd.Timestamp(STRICT_START))
    & (aligned_test_dates <= pd.Timestamp(STRICT_END))
]

if len(aligned_test_dates) == 0:
    raise ValueError("No aligned evaluation dates found.")

uniform_vec = np.ones(len(CLASS_LABELS)) / len(CLASS_LABELS)

p20_uniform = make_constant_probability_frame(
    aligned_test_dates,
    uniform_vec,
    model_name="uniform_probability_20d",
)

p60_uniform = make_constant_probability_frame(
    aligned_test_dates,
    uniform_vec,
    model_name="uniform_probability_60d",
)

print("ETF return panel:", etf_return_path)
print("Base ETF return shape:", base_etf_returns.shape)
print("20d probability file:", p20_path)
print("60d probability file:", p60_path)
print("Aligned evaluation dates:", len(aligned_test_dates), aligned_test_dates.min().date(), "to", aligned_test_dates.max().date())

# ============================================================
# 6. Build return panels for universe variants
# ============================================================

def build_return_panel_for_universe(base_returns, universe_key):
    if universe_key == "FULL":
        asset_list = ["0050", "006208", "00692", "00881"]
        ret = base_returns[asset_list].copy()
        description = "Full universe: 0050, 006208, 00692, 00881"
    elif universe_key == "REMOVE_0050":
        asset_list = ["006208", "00692", "00881"]
        ret = base_returns[asset_list].copy()
        description = "Reduced universe excluding 0050"
    elif universe_key == "REMOVE_006208":
        asset_list = ["0050", "00692", "00881"]
        ret = base_returns[asset_list].copy()
        description = "Reduced universe excluding 006208"
    elif universe_key == "SYNTH_TW50":
        ret = pd.DataFrame(index=base_returns.index)
        ret["TW50_SYN"] = 0.5 * base_returns["0050"] + 0.5 * base_returns["006208"]
        ret["00692"] = base_returns["00692"]
        ret["00881"] = base_returns["00881"]
        asset_list = ["TW50_SYN", "00692", "00881"]
        description = "Synthetic Taiwan 50 sleeve: 0.5*0050 + 0.5*006208, plus 00692 and 00881"
    else:
        raise ValueError(f"Unknown universe_key: {universe_key}")

    ret = ret.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return ret, asset_list, description

UNIVERSE_VARIANTS = [
    ("FULL", "Full universe"),
    ("REMOVE_0050", "Remove 0050"),
    ("REMOVE_006208", "Remove 006208"),
    ("SYNTH_TW50", "Synthetic Taiwan 50 sleeve"),
]

# ============================================================
# 7. Run original dynamic AURORA and constant-lambda grid
# ============================================================

print("\n" + "=" * 90)
print("Running original dynamic AURORA and exposure-matched constant-lambda grid")
print("=" * 90)

metric_rows = []
exposure_rows = []
return_frames = []
weight_frames = []
diag_frames = []
feature_frames = []
grid_rows = []

return_series_by_label = {}
weight_by_label = {}
diag_by_label = {}
feature_by_label = {}
config_by_label = {}
asset_list_by_label = {}

def make_config(name, changes=None):
    cfg = copy.deepcopy(BASE_UAMV_B_CONFIG)
    cfg["strategy_name"] = name
    if changes:
        for k, v in changes.items():
            cfg[k] = v
    return cfg

def record_result(label, policy_name, analysis_group, ret_df, weight_df, diag_df, feat_df, config, asset_list, extra=None):
    metrics = performance_metrics(ret_df)
    exposure = exposure_diagnostics(
        weight_df=weight_df,
        return_df=ret_df,
        label=label,
        config=config,
        asset_list=asset_list,
    )

    metrics.update({
        "run_id": RUN_ID,
        "strategy_control": label,
        "policy_name": policy_name,
        "analysis_group": analysis_group,
        "asset_universe": ",".join(asset_list),
        "risk_aversion_mode": config.get("risk_aversion_mode", "dynamic"),
        "constant_lambda": config.get("constant_lambda", np.nan),
        "max_cash_weight_config": config.get("max_cash_weight", np.nan),
    })

    if extra:
        metrics.update(extra)

    metric_rows.append(metrics)
    exposure_rows.append(exposure)

    ret_out = ret_df.copy()
    ret_out["strategy_control"] = label
    ret_out["policy_name"] = policy_name
    ret_out["analysis_group"] = analysis_group
    ret_out["asset_universe"] = ",".join(asset_list)
    return_frames.append(ret_out)

    w_out = weight_df.copy()
    w_out.insert(0, "strategy_control", label)
    w_out.insert(0, "policy_name", policy_name)
    w_out.insert(0, "analysis_group", analysis_group)
    w_out.insert(0, "asset_universe", ",".join(asset_list))
    weight_frames.append(w_out)

    d_out = diag_df.copy()
    d_out["strategy_control"] = label
    d_out["policy_name"] = policy_name
    d_out["analysis_group"] = analysis_group
    d_out["asset_universe"] = ",".join(asset_list)
    diag_frames.append(d_out)

    f_out = feat_df.copy()
    f_out["strategy_control"] = label
    f_out["policy_name"] = policy_name
    f_out["analysis_group"] = analysis_group
    f_out["asset_universe"] = ",".join(asset_list)
    feature_frames.append(f_out)

    return_series_by_label[label] = ret_df["net_return"].copy()
    weight_by_label[label] = weight_df.copy()
    diag_by_label[label] = diag_df.copy()
    feature_by_label[label] = feat_df.copy()
    config_by_label[label] = copy.deepcopy(config)
    asset_list_by_label[label] = list(asset_list)

    ret_df.to_parquet(RETURN_DIR / f"returns_{safe_name(policy_name)}.parquet")
    ret_df.to_csv(RETURN_DIR / f"returns_{safe_name(policy_name)}.csv")
    weight_df.to_parquet(WEIGHT_DIR / f"weights_{safe_name(policy_name)}.parquet")
    weight_df.to_csv(WEIGHT_DIR / f"weights_{safe_name(policy_name)}.csv")

    print(
        f"{label}: total={metrics['total_return']:.4f}, "
        f"Sharpe={metrics['sharpe_ratio']:.4f}, "
        f"Sortino={metrics['sortino_ratio']:.4f}, "
        f"MaxDD={metrics['max_drawdown']:.4f}, "
        f"AvgCash={exposure['avg_cash_weight']:.4f}, "
        f"CapDays={exposure['pct_days_at_cash_cap']:.4f}"
    )

full_returns, full_assets, full_desc = build_return_panel_for_universe(base_etf_returns, "FULL")

# Original dynamic AURORA, same as reported 60% cash-cap diagnostic setting.
dynamic_cfg = make_config("AURORA18_DYNAMIC_ORIGINAL", {"risk_aversion_mode": "dynamic"})

dyn_signal_w, dyn_diag, dyn_feat = build_uamv_signal_weights(
    p20=p20_test_latest,
    p60=p60_test_latest,
    etf_returns=full_returns,
    config=dynamic_cfg,
    evaluation_dates=aligned_test_dates,
    asset_list=full_assets,
)

dyn_ret, dyn_daily_w = backtest_on_fixed_index(
    policy_name="AURORA18_DYNAMIC_ORIGINAL",
    signal_weights=dyn_signal_w,
    etf_returns=full_returns,
    evaluation_index=aligned_test_dates,
    asset_list=full_assets,
)

record_result(
    label="Original dynamic AURORA",
    policy_name="AURORA18_DYNAMIC_ORIGINAL",
    analysis_group="constant_lambda_exposure_match",
    ret_df=dyn_ret,
    weight_df=dyn_daily_w,
    diag_df=dyn_diag,
    feat_df=dyn_feat,
    config=dynamic_cfg,
    asset_list=full_assets,
    extra={"universe_description": full_desc},
)

TARGET_AVG_CASH = float(dyn_daily_w[CASH_COL].mean())
print(f"\nTarget average cash from original dynamic AURORA = {TARGET_AVG_CASH:.6f}")

# Constant-lambda grid, using original probability sequence and regime tilt.
for lam in CONSTANT_LAMBDA_GRID:
    cfg = make_config(
        f"AURORA18_CONST_LAMBDA_ORIGPROB_{lam:g}",
        {
            "risk_aversion_mode": "constant_lambda",
            "constant_lambda": float(lam),
            "regime_tilt_strength": BASE_UAMV_B_CONFIG["regime_tilt_strength"],
            "uncertainty_risk_multiplier": BASE_UAMV_B_CONFIG["uncertainty_risk_multiplier"],
            "bearish_risk_multiplier": BASE_UAMV_B_CONFIG["bearish_risk_multiplier"],
        },
    )

    signal_w, diag_df, feat_df = build_uamv_signal_weights(
        p20=p20_test_latest,
        p60=p60_test_latest,
        etf_returns=full_returns,
        config=cfg,
        evaluation_dates=aligned_test_dates,
        asset_list=full_assets,
    )

    ret_df, daily_w = backtest_on_fixed_index(
        policy_name=f"AURORA18_CONST_LAMBDA_ORIGPROB_{safe_name(lam)}",
        signal_weights=signal_w,
        etf_returns=full_returns,
        evaluation_index=aligned_test_dates,
        asset_list=full_assets,
    )

    metrics = performance_metrics(ret_df)
    exposure = exposure_diagnostics(daily_w, ret_df, f"lambda_{lam:g}", cfg, full_assets)

    grid_rows.append({
        "grid_family": "constant_lambda_original_probabilities",
        "lambda_value": float(lam),
        "avg_cash_weight": exposure["avg_cash_weight"],
        "cash_match_error_abs": abs(exposure["avg_cash_weight"] - TARGET_AVG_CASH),
        "pct_days_at_cash_cap": exposure["pct_days_at_cash_cap"],
        "total_return": metrics["total_return"],
        "sharpe_ratio": metrics["sharpe_ratio"],
        "sortino_ratio": metrics["sortino_ratio"],
        "max_drawdown": metrics["max_drawdown"],
        "calmar_ratio": metrics["calmar_ratio"],
    })

# Constant-lambda no-probability grid: uniform probabilities, no entropy/bearish terms, no regime tilt.
for lam in CONSTANT_LAMBDA_GRID:
    cfg = make_config(
        f"AURORA18_CONST_LAMBDA_NOPROB_{lam:g}",
        {
            "risk_aversion_mode": "constant_lambda",
            "constant_lambda": float(lam),
            "uncertainty_risk_multiplier": 0.0,
            "bearish_risk_multiplier": 0.0,
            "regime_tilt_strength": 0.0,
        },
    )

    signal_w, diag_df, feat_df = build_uamv_signal_weights(
        p20=p20_uniform,
        p60=p60_uniform,
        etf_returns=full_returns,
        config=cfg,
        evaluation_dates=aligned_test_dates,
        asset_list=full_assets,
    )

    ret_df, daily_w = backtest_on_fixed_index(
        policy_name=f"AURORA18_CONST_LAMBDA_NOPROB_{safe_name(lam)}",
        signal_weights=signal_w,
        etf_returns=full_returns,
        evaluation_index=aligned_test_dates,
        asset_list=full_assets,
    )

    metrics = performance_metrics(ret_df)
    exposure = exposure_diagnostics(daily_w, ret_df, f"lambda_{lam:g}", cfg, full_assets)

    grid_rows.append({
        "grid_family": "constant_lambda_no_probability",
        "lambda_value": float(lam),
        "avg_cash_weight": exposure["avg_cash_weight"],
        "cash_match_error_abs": abs(exposure["avg_cash_weight"] - TARGET_AVG_CASH),
        "pct_days_at_cash_cap": exposure["pct_days_at_cash_cap"],
        "total_return": metrics["total_return"],
        "sharpe_ratio": metrics["sharpe_ratio"],
        "sortino_ratio": metrics["sortino_ratio"],
        "max_drawdown": metrics["max_drawdown"],
        "calmar_ratio": metrics["calmar_ratio"],
    })

lambda_grid_df = pd.DataFrame(grid_rows)

best_origprob = (
    lambda_grid_df[lambda_grid_df["grid_family"] == "constant_lambda_original_probabilities"]
    .sort_values(["cash_match_error_abs", "lambda_value"])
    .iloc[0]
)

best_noprob = (
    lambda_grid_df[lambda_grid_df["grid_family"] == "constant_lambda_no_probability"]
    .sort_values(["cash_match_error_abs", "lambda_value"])
    .iloc[0]
)

print("\nBest constant-lambda original-probability match:")
print(best_origprob.to_string())
print("\nBest constant-lambda no-probability match:")
print(best_noprob.to_string())

# Record best constant-lambda original-probability control.
for best_row, label_prefix, p20_use, p60_use, changes, analysis_label in [
    (
        best_origprob,
        "Exposure-matched constant-lambda AURORA",
        p20_test_latest,
        p60_test_latest,
        {
            "risk_aversion_mode": "constant_lambda",
            "constant_lambda": float(best_origprob["lambda_value"]),
            "regime_tilt_strength": BASE_UAMV_B_CONFIG["regime_tilt_strength"],
        },
        "constant_lambda_exposure_match",
    ),
    (
        best_noprob,
        "Exposure-matched constant-lambda no-probability AURORA",
        p20_uniform,
        p60_uniform,
        {
            "risk_aversion_mode": "constant_lambda",
            "constant_lambda": float(best_noprob["lambda_value"]),
            "uncertainty_risk_multiplier": 0.0,
            "bearish_risk_multiplier": 0.0,
            "regime_tilt_strength": 0.0,
        },
        "constant_lambda_exposure_match",
    ),
]:
    lam = float(best_row["lambda_value"])
    cfg = make_config(
        f"AURORA18_{safe_name(label_prefix)}_{lam:g}",
        changes,
    )

    signal_w, diag_df, feat_df = build_uamv_signal_weights(
        p20=p20_use,
        p60=p60_use,
        etf_returns=full_returns,
        config=cfg,
        evaluation_dates=aligned_test_dates,
        asset_list=full_assets,
    )

    ret_df, daily_w = backtest_on_fixed_index(
        policy_name=f"AURORA18_{safe_name(label_prefix)}_{safe_name(lam)}",
        signal_weights=signal_w,
        etf_returns=full_returns,
        evaluation_index=aligned_test_dates,
        asset_list=full_assets,
    )

    record_result(
        label=label_prefix,
        policy_name=f"AURORA18_{safe_name(label_prefix)}_{safe_name(lam)}",
        analysis_group=analysis_label,
        ret_df=ret_df,
        weight_df=daily_w,
        diag_df=diag_df,
        feat_df=feat_df,
        config=cfg,
        asset_list=full_assets,
        extra={
            "selected_lambda": lam,
            "target_avg_cash": TARGET_AVG_CASH,
            "cash_match_error_abs": abs(float(daily_w[CASH_COL].mean()) - TARGET_AVG_CASH),
            "universe_description": full_desc,
        },
    )

# Add static exposure-matched cash baseline for context.
static_w = pd.DataFrame(index=aligned_test_dates, columns=all_assets_for(full_assets), dtype=float)
static_cash = TARGET_AVG_CASH
static_risky_each = (1.0 - static_cash) / len(full_assets)
for asset in full_assets:
    static_w[asset] = static_risky_each
static_w[CASH_COL] = static_cash
static_w = normalize_rows(static_w, full_assets)

static_ret, static_daily_w = backtest_on_fixed_index(
    policy_name="AURORA18_STATIC_EXPOSURE_MATCHED_CASH",
    signal_weights=static_w,
    etf_returns=full_returns,
    evaluation_index=aligned_test_dates,
    asset_list=full_assets,
)

static_cfg = make_config(
    "AURORA18_STATIC_EXPOSURE_MATCHED_CASH",
    {
        "risk_aversion_mode": "static_baseline",
        "constant_lambda": np.nan,
    },
)

empty_diag = dyn_diag.copy()
empty_diag["risk_aversion"] = np.nan
empty_diag["optimization_status"] = "static_baseline"

record_result(
    label="Static exposure-matched cash baseline",
    policy_name="AURORA18_STATIC_EXPOSURE_MATCHED_CASH",
    analysis_group="constant_lambda_exposure_match",
    ret_df=static_ret,
    weight_df=static_daily_w,
    diag_df=empty_diag,
    feat_df=dyn_feat,
    config=static_cfg,
    asset_list=full_assets,
    extra={
        "target_avg_cash": TARGET_AVG_CASH,
        "static_cash_weight": static_cash,
        "universe_description": full_desc,
    },
)

# ============================================================
# 8. Reduced-universe 0050/006208 sensitivity
# ============================================================

print("\n" + "=" * 90)
print("Running reduced-universe 0050/006208 sensitivity")
print("=" * 90)

for universe_key, universe_label in UNIVERSE_VARIANTS:
    ret_panel, asset_list, desc = build_return_panel_for_universe(base_etf_returns, universe_key)

    cfg = make_config(
        f"AURORA18_REDUCED_{universe_key}",
        {
            "risk_aversion_mode": "dynamic",
            "max_cash_weight": CASH_CAP_REPORTED,
        },
    )

    signal_w, diag_df, feat_df = build_uamv_signal_weights(
        p20=p20_test_latest,
        p60=p60_test_latest,
        etf_returns=ret_panel,
        config=cfg,
        evaluation_dates=aligned_test_dates,
        asset_list=asset_list,
    )

    ret_df, daily_w = backtest_on_fixed_index(
        policy_name=f"AURORA18_REDUCED_{universe_key}",
        signal_weights=signal_w,
        etf_returns=ret_panel,
        evaluation_index=aligned_test_dates,
        asset_list=asset_list,
    )

    cond_stats = condition_number_stats(
        etf_returns=ret_panel,
        evaluation_index=aligned_test_dates,
        asset_list=asset_list,
        lookback=int(cfg["lookback_cov"]),
    )

    record_result(
        label=f"Reduced-universe: {universe_label}",
        policy_name=f"AURORA18_REDUCED_{universe_key}",
        analysis_group="reduced_universe_sensitivity",
        ret_df=ret_df,
        weight_df=daily_w,
        diag_df=diag_df,
        feat_df=feat_df,
        config=cfg,
        asset_list=asset_list,
        extra={
            "universe_key": universe_key,
            "universe_description": desc,
            **cond_stats,
        },
    )

# ============================================================
# 9. Bootstrap comparisons for constant-lambda controls
# ============================================================

print("\n" + "=" * 90)
print("Running bootstrap comparisons for constant-lambda controls")
print("=" * 90)

bootstrap_comparators = [
    "Exposure-matched constant-lambda AURORA",
    "Exposure-matched constant-lambda no-probability AURORA",
    "Static exposure-matched cash baseline",
]

bootstrap_frames = []

for comp_label in bootstrap_comparators:
    if "Original dynamic AURORA" not in return_series_by_label or comp_label not in return_series_by_label:
        print("Skipping bootstrap; missing:", comp_label)
        continue

    print("Bootstrap: Original dynamic AURORA vs", comp_label)

    boot_df = paired_circular_block_bootstrap(
        strategy_returns=return_series_by_label["Original dynamic AURORA"],
        comparator_returns=return_series_by_label[comp_label],
        n_rep=BOOTSTRAP_REPLICATIONS,
        block_length=BOOTSTRAP_BLOCK_LENGTH,
        seed=BOOTSTRAP_RANDOM_SEED + len(bootstrap_frames),
    )

    boot_df.insert(0, "comparison", f"Original dynamic AURORA vs {comp_label}")
    boot_df.insert(1, "strategy_control", "Original dynamic AURORA")
    boot_df.insert(2, "comparator_control", comp_label)

    bootstrap_frames.append(boot_df)

constant_lambda_bootstrap_df = pd.concat(bootstrap_frames, ignore_index=True) if bootstrap_frames else pd.DataFrame()

# ============================================================
# 10. Build mechanism-attribution figure
# ============================================================

print("\n" + "=" * 90)
print("Creating mechanism-attribution figure")
print("=" * 90)

mechanism_label = "Original dynamic AURORA"
mechanism_ret = dyn_ret.copy()
mechanism_w = dyn_daily_w.copy()
mechanism_diag = dyn_diag.copy()
mechanism_feat = dyn_feat.copy()

fig_df = pd.DataFrame(index=aligned_test_dates)
fig_df["drawdown"] = mechanism_ret["drawdown"]
fig_df["cash_weight"] = mechanism_w[CASH_COL]
fig_df["risk_aversion"] = mechanism_diag["risk_aversion"]
fig_df["normalized_entropy"] = mechanism_feat["normalized_entropy"]
fig_df["p_bearish"] = mechanism_feat["p_bearish"]
fig_df["cash_cap"] = CASH_CAP_REPORTED
fig_df["at_cash_cap"] = (fig_df["cash_weight"] >= CASH_CAP_REPORTED - 1e-8).astype(int)

fig_path_png = PLOT_DIR / "figure_S4_mechanism_attribution.png"
fig_path_pdf = PLOT_DIR / "figure_S4_mechanism_attribution.pdf"
fig_global_png = FIGURE_DIR / f"figure_S4_mechanism_attribution_{RUN_ID}.png"
fig_global_pdf = FIGURE_DIR / f"figure_S4_mechanism_attribution_{RUN_ID}.pdf"

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
})

fig, axes = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(11, 9),
    sharex=True,
    gridspec_kw={"height_ratios": [1.1, 1.1, 1.0, 1.1]},
)

axes[0].plot(fig_df.index, fig_df["drawdown"], color="#1f77b4", linewidth=1.8)
axes[0].axhline(0.0, color="black", linewidth=0.8)
axes[0].set_ylabel("Drawdown")
axes[0].set_title("A. AURORA cash cap 60% drawdown")
axes[0].grid(True, alpha=0.25)

axes[1].plot(fig_df.index, fig_df["normalized_entropy"], color="#9467bd", linewidth=1.6, label=r"Entropy $U_t$")
axes[1].plot(fig_df.index, fig_df["p_bearish"], color="#d62728", linewidth=1.6, label=r"Bearish probability $B_t$")
axes[1].set_ylabel("Probability / entropy")
axes[1].set_title("B. Probability-derived uncertainty and bearish-risk inputs")
axes[1].legend(loc="upper left", ncol=2)
axes[1].grid(True, alpha=0.25)

axes[2].plot(fig_df.index, fig_df["risk_aversion"], color="#ff7f0e", linewidth=1.8)
axes[2].set_ylabel(r"$\lambda_t$")
axes[2].set_title("C. Effective state-dependent risk aversion")
axes[2].grid(True, alpha=0.25)

axes[3].plot(fig_df.index, fig_df["cash_weight"], color="#2ca02c", linewidth=1.8, label="Cash weight")
axes[3].axhline(CASH_CAP_REPORTED, color="#d62728", linewidth=1.2, linestyle="--", label="Cash cap = 0.60")
axes[3].fill_between(
    fig_df.index,
    0,
    fig_df["at_cash_cap"] * CASH_CAP_REPORTED,
    color="#d62728",
    alpha=0.10,
    label="At cash cap",
)
axes[3].set_ylabel("Cash weight")
axes[3].set_title("D. Cash allocation and cash-cap binding")
axes[3].set_ylim(-0.02, 0.82)
axes[3].legend(loc="upper left", ncol=3)
axes[3].grid(True, alpha=0.25)

axes[3].set_xlabel("Date")

fig.suptitle(
    "Figure S4. Mechanism-attribution diagnostics for AURORA cash cap 60%",
    fontsize=13,
    fontweight="bold",
    y=0.995,
)

fig.tight_layout(rect=[0, 0, 1, 0.975])
fig.savefig(fig_path_png, dpi=300)
fig.savefig(fig_path_pdf)
fig_global_png.write_bytes(fig_path_png.read_bytes())
fig_global_pdf.write_bytes(fig_path_pdf.read_bytes())
plt.close(fig)

fig_df.to_csv(DIAG_DIR / "figure_S4_mechanism_attribution_data.csv")

print("Saved mechanism figure:")
print(fig_path_png)
print(fig_path_pdf)

# ============================================================
# 11. Export tables
# ============================================================

print("\n" + "=" * 90)
print("Exporting Notebook 18 tables")
print("=" * 90)

metrics_df = pd.DataFrame(metric_rows)
exposure_df = pd.DataFrame(exposure_rows)

# Table S35: constant-lambda exposure match summary.
s35_labels = [
    "Original dynamic AURORA",
    "Exposure-matched constant-lambda AURORA",
    "Exposure-matched constant-lambda no-probability AURORA",
    "Static exposure-matched cash baseline",
]

s35 = metrics_df[metrics_df["strategy_control"].isin(s35_labels)].copy()
s35_exp = exposure_df[exposure_df["strategy_control"].isin(s35_labels)].copy()

s35 = s35.merge(
    s35_exp[
        [
            "strategy_control",
            "avg_cash_weight",
            "median_cash_weight",
            "pct_days_at_cash_cap",
            "avg_equity_exposure",
            "avg_00881_weight",
            "total_turnover",
            "total_transaction_cost",
        ]
    ],
    on="strategy_control",
    how="left",
)

s35["cash_match_error_abs"] = (s35["avg_cash_weight"] - TARGET_AVG_CASH).abs()

s35_order = {label: i for i, label in enumerate(s35_labels)}
s35["display_order"] = s35["strategy_control"].map(s35_order)
s35 = s35.sort_values("display_order")

s35_cols = [
    "strategy_control",
    "constant_lambda",
    "total_return",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "avg_cash_weight",
    "pct_days_at_cash_cap",
    "avg_equity_exposure",
    "cash_match_error_abs",
    "total_turnover",
    "total_transaction_cost",
]

s35 = s35[[c for c in s35_cols if c in s35.columns]].copy()

# Table S36: reduced-universe performance.
s36 = metrics_df[metrics_df["analysis_group"] == "reduced_universe_sensitivity"].copy()
s36_exp = exposure_df[exposure_df["strategy_control"].isin(s36["strategy_control"].tolist())].copy()

s36 = s36.merge(
    s36_exp[
        [
            "strategy_control",
            "avg_cash_weight",
            "pct_days_at_cash_cap",
            "avg_equity_exposure",
            "avg_00881_weight",
            "total_turnover",
            "total_transaction_cost",
        ]
    ],
    on="strategy_control",
    how="left",
)

s36_cols = [
    "strategy_control",
    "universe_key",
    "universe_description",
    "asset_universe",
    "n_days",
    "total_return",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "avg_cash_weight",
    "pct_days_at_cash_cap",
    "avg_equity_exposure",
    "median_cov_condition_number",
    "p90_cov_condition_number",
]

s36 = s36[[c for c in s36_cols if c in s36.columns]].copy()

# Table S37: reduced-universe exposure diagnostics.
s37 = exposure_df[
    exposure_df["strategy_control"].isin(
        metrics_df[metrics_df["analysis_group"] == "reduced_universe_sensitivity"]["strategy_control"].tolist()
    )
].copy()

cond_cols = metrics_df[
    metrics_df["analysis_group"] == "reduced_universe_sensitivity"
][
    [
        "strategy_control",
        "universe_key",
        "universe_description",
        "median_cov_condition_number",
        "p90_cov_condition_number",
        "max_cov_condition_number",
    ]
].copy()

s37 = s37.merge(cond_cols, on="strategy_control", how="left")

s37_cols = [
    "strategy_control",
    "universe_key",
    "asset_universe",
    "avg_cash_weight",
    "median_cash_weight",
    "pct_days_at_cash_cap",
    "avg_equity_exposure",
    "avg_0050_weight",
    "avg_006208_weight",
    "avg_TW50_SYN_weight",
    "avg_00692_weight",
    "avg_00881_weight",
    "avg_max_risky_asset_weight",
    "total_turnover",
    "total_transaction_cost",
    "median_cov_condition_number",
    "p90_cov_condition_number",
]

for c in ["avg_0050_weight", "avg_006208_weight", "avg_TW50_SYN_weight", "avg_00692_weight", "avg_00881_weight"]:
    if c not in s37.columns:
        s37[c] = np.nan

s37 = s37[[c for c in s37_cols if c in s37.columns]].copy()

# Table S38: constant-lambda bootstrap.
s38 = constant_lambda_bootstrap_df.copy()

# Table S39: constant-lambda grid search.
s39 = lambda_grid_df.copy().sort_values(["grid_family", "cash_match_error_abs", "lambda_value"])

outputs = {
    "table_S35_constant_lambda_exposure_match.csv": s35,
    "table_S36_reduced_universe_sensitivity.csv": s36,
    "table_S37_reduced_universe_exposure_diagnostics.csv": s37,
    "table_S38_constant_lambda_bootstrap.csv": s38,
    "table_S39_constant_lambda_grid_search.csv": s39,
}

for fname, df in outputs.items():
    local_path = TABLE_RUN_DIR / fname
    global_path = TABLE_DIR / f"{Path(fname).stem}_{RUN_ID}.csv"
    df.to_csv(local_path, index=False)
    df.to_csv(global_path, index=False)
    print("Saved:", local_path)
    print("Saved:", global_path)

def save_rounded(df, local_name, global_name, digits=6):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(digits)
    out.to_csv(TABLE_RUN_DIR / local_name, index=False)
    out.to_csv(TABLE_DIR / global_name, index=False)
    return out

s35_round = save_rounded(
    s35,
    "table_S35_constant_lambda_exposure_match_rounded.csv",
    f"table_S35_constant_lambda_exposure_match_rounded_{RUN_ID}.csv",
)

s36_round = save_rounded(
    s36,
    "table_S36_reduced_universe_sensitivity_rounded.csv",
    f"table_S36_reduced_universe_sensitivity_rounded_{RUN_ID}.csv",
)

s37_round = save_rounded(
    s37,
    "table_S37_reduced_universe_exposure_diagnostics_rounded.csv",
    f"table_S37_reduced_universe_exposure_diagnostics_rounded_{RUN_ID}.csv",
)

s38_round = save_rounded(
    s38,
    "table_S38_constant_lambda_bootstrap_rounded.csv",
    f"table_S38_constant_lambda_bootstrap_rounded_{RUN_ID}.csv",
)

s39_round = save_rounded(
    s39,
    "table_S39_constant_lambda_grid_search_rounded.csv",
    f"table_S39_constant_lambda_grid_search_rounded_{RUN_ID}.csv",
)

# Export combined returns, weights, diagnostics.
all_returns_df = pd.concat(return_frames, axis=0).sort_index() if return_frames else pd.DataFrame()
all_weights_df = pd.concat(weight_frames, axis=0).sort_index() if weight_frames else pd.DataFrame()
all_diag_df = pd.concat(diag_frames, axis=0).sort_index() if diag_frames else pd.DataFrame()
all_features_df = pd.concat(feature_frames, axis=0).sort_index() if feature_frames else pd.DataFrame()

all_returns_df.to_parquet(RETURN_DIR / "all_notebook18_returns.parquet")
all_returns_df.to_csv(RETURN_DIR / "all_notebook18_returns.csv")

all_weights_df.to_parquet(WEIGHT_DIR / "all_notebook18_weights.parquet")
all_weights_df.to_csv(WEIGHT_DIR / "all_notebook18_weights.csv")

all_diag_df.to_parquet(DIAG_DIR / "all_notebook18_diagnostics.parquet")
all_diag_df.to_csv(DIAG_DIR / "all_notebook18_diagnostics.csv")

all_features_df.to_parquet(DIAG_DIR / "all_notebook18_probability_features.parquet")
all_features_df.to_csv(DIAG_DIR / "all_notebook18_probability_features.csv")

lambda_grid_df.to_csv(DIAG_DIR / "constant_lambda_grid_search_full.csv", index=False)

# ============================================================
# 12. Interpretation helper
# ============================================================

interpretation_rows = []

def add_interpretation(question, finding, evidence):
    interpretation_rows.append({
        "diagnostic_question": question,
        "finding": finding,
        "evidence": evidence,
    })

def metric_row_by_label(label):
    rows = metrics_df[metrics_df["strategy_control"] == label]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

def exposure_row_by_label(label):
    rows = exposure_df[exposure_df["strategy_control"] == label]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()

orig_m = metric_row_by_label("Original dynamic AURORA")
orig_e = exposure_row_by_label("Original dynamic AURORA")
cl_m = metric_row_by_label("Exposure-matched constant-lambda AURORA")
cl_e = exposure_row_by_label("Exposure-matched constant-lambda AURORA")
clnp_m = metric_row_by_label("Exposure-matched constant-lambda no-probability AURORA")
clnp_e = exposure_row_by_label("Exposure-matched constant-lambda no-probability AURORA")

if orig_m and cl_m and orig_e and cl_e:
    add_interpretation(
        "Does dynamic AURORA clearly outperform exposure-matched constant-lambda AURORA?",
        "Compare Table S35 and Table S38",
        (
            f"Original dynamic: total={orig_m['total_return']:.4f}, Sharpe={orig_m['sharpe_ratio']:.4f}, "
            f"Sortino={orig_m['sortino_ratio']:.4f}, MaxDD={orig_m['max_drawdown']:.4f}, "
            f"AvgCash={orig_e['avg_cash_weight']:.4f}. "
            f"Constant-lambda: total={cl_m['total_return']:.4f}, Sharpe={cl_m['sharpe_ratio']:.4f}, "
            f"Sortino={cl_m['sortino_ratio']:.4f}, MaxDD={cl_m['max_drawdown']:.4f}, "
            f"AvgCash={cl_e['avg_cash_weight']:.4f}."
        ),
    )

if orig_m and clnp_m and orig_e and clnp_e:
    add_interpretation(
        "Does dynamic AURORA clearly outperform exposure-matched no-probability constant-lambda AURORA?",
        "Compare Table S35 and Table S38",
        (
            f"Original dynamic: total={orig_m['total_return']:.4f}, Sharpe={orig_m['sharpe_ratio']:.4f}, "
            f"Sortino={orig_m['sortino_ratio']:.4f}, MaxDD={orig_m['max_drawdown']:.4f}, "
            f"AvgCash={orig_e['avg_cash_weight']:.4f}. "
            f"No-probability constant-lambda: total={clnp_m['total_return']:.4f}, "
            f"Sharpe={clnp_m['sharpe_ratio']:.4f}, Sortino={clnp_m['sortino_ratio']:.4f}, "
            f"MaxDD={clnp_m['max_drawdown']:.4f}, AvgCash={clnp_e['avg_cash_weight']:.4f}."
        ),
    )

ru = metrics_df[metrics_df["analysis_group"] == "reduced_universe_sensitivity"].copy()
if not ru.empty:
    best_ru = ru.sort_values("sharpe_ratio", ascending=False).iloc[0]
    add_interpretation(
        "Are results sensitive to the near-duplicate 0050/006208 exposures?",
        "Inspect reduced-universe sensitivity",
        (
            f"Best reduced-universe Sharpe row: {best_ru['strategy_control']}, "
            f"total={best_ru['total_return']:.4f}, Sharpe={best_ru['sharpe_ratio']:.4f}, "
            f"Sortino={best_ru['sortino_ratio']:.4f}, MaxDD={best_ru['max_drawdown']:.4f}."
        ),
    )

interpretation_df = pd.DataFrame(interpretation_rows)
interpretation_df.to_csv(TABLE_RUN_DIR / "notebook18_interpretation_helper.csv", index=False)
interpretation_df.to_csv(TABLE_DIR / f"notebook18_interpretation_helper_{RUN_ID}.csv", index=False)

# ============================================================
# 13. Validation report and manifest
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "18_AURORA_exposure_matched_lambda_reduced_universe_mechanism_figure",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Addresses reviewer-requested diagnostics: exposure-matched constant-lambda optimizer, "
        "reduced-universe 0050/006208 sensitivity, and mechanism-attribution figure."
    ),
    "input_paths": {
        "etf_return_panel": str(etf_return_path),
        "notebook08_or_allocation_input_index": str(NOTEBOOK08_INPUT_INDEX),
        "probability_20d_path": str(p20_path),
        "probability_60d_path": str(p60_path),
    },
    "evaluation_window": {
        "requested_start": STRICT_START,
        "requested_end": STRICT_END,
        "n_days": int(len(aligned_test_dates)),
        "actual_start": str(aligned_test_dates.min().date()),
        "actual_end": str(aligned_test_dates.max().date()),
    },
    "constant_lambda_grid": CONSTANT_LAMBDA_GRID.tolist(),
    "target_avg_cash": TARGET_AVG_CASH,
    "selected_constant_lambda_original_probability": float(best_origprob["lambda_value"]),
    "selected_constant_lambda_no_probability": float(best_noprob["lambda_value"]),
    "universe_variants": [
        {
            "key": key,
            "label": label,
            "description": build_return_panel_for_universe(base_etf_returns, key)[2],
            "assets": build_return_panel_for_universe(base_etf_returns, key)[1],
        }
        for key, label in UNIVERSE_VARIANTS
    ],
    "bootstrap": {
        "replications": BOOTSTRAP_REPLICATIONS,
        "block_length": BOOTSTRAP_BLOCK_LENGTH,
        "random_seed": BOOTSTRAP_RANDOM_SEED,
        "ci_type": "percentile",
        "bootstrap_type": "paired circular block bootstrap",
    },
    "base_config": BASE_UAMV_B_CONFIG,
    "has_scipy_optimizer": HAS_SCIPY,
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_DIR),
        "weights": str(WEIGHT_DIR),
        "diagnostics": str(DIAG_DIR),
        "plots": str(PLOT_DIR),
        "table_S35": str(TABLE_RUN_DIR / "table_S35_constant_lambda_exposure_match.csv"),
        "table_S36": str(TABLE_RUN_DIR / "table_S36_reduced_universe_sensitivity.csv"),
        "table_S37": str(TABLE_RUN_DIR / "table_S37_reduced_universe_exposure_diagnostics.csv"),
        "table_S38": str(TABLE_RUN_DIR / "table_S38_constant_lambda_bootstrap.csv"),
        "table_S39": str(TABLE_RUN_DIR / "table_S39_constant_lambda_grid_search.csv"),
        "figure_S4_png": str(fig_path_png),
        "figure_S4_pdf": str(fig_path_pdf),
    },
    "educational_note": (
        "This notebook performs research backtests only and does not provide personalized financial advice."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK18_validation_report.json"
validation_report_global_path = REPORT_DIR / f"NOTEBOOK18_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "NOTEBOOK18_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"NOTEBOOK18_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 14. Final summary
# ============================================================

print("\n" + "=" * 90)
print("AURORA-TWETF NOTEBOOK 18 COMPLETE")
print("=" * 90)
print("Run ID:", RUN_ID)
print("Run root:", RUN_ROOT)
print("Table S35:", TABLE_RUN_DIR / "table_S35_constant_lambda_exposure_match.csv")
print("Table S36:", TABLE_RUN_DIR / "table_S36_reduced_universe_sensitivity.csv")
print("Table S37:", TABLE_RUN_DIR / "table_S37_reduced_universe_exposure_diagnostics.csv")
print("Table S38:", TABLE_RUN_DIR / "table_S38_constant_lambda_bootstrap.csv")
print("Table S39:", TABLE_RUN_DIR / "table_S39_constant_lambda_grid_search.csv")
print("Figure S4 PNG:", fig_path_png)
print("Figure S4 PDF:", fig_path_pdf)
print("All returns:", RETURN_DIR / "all_notebook18_returns.parquet")
print("All weights:", WEIGHT_DIR / "all_notebook18_weights.parquet")
print("All diagnostics:", DIAG_DIR / "all_notebook18_diagnostics.parquet")
print("Validation report:", validation_report_path)
print("Manifest:", manifest_path)
print("=" * 90)

print("\nRounded Table S35 preview:")
print(s35_round.to_string(index=False))

print("\nRounded Table S36 preview:")
print(s36_round.to_string(index=False))

print("\nRounded Table S37 preview:")
print(s37_round.to_string(index=False))

print("\nRounded Table S38 preview:")
if not s38_round.empty:
    print(s38_round.to_string(index=False))
else:
    print("No bootstrap rows.")

print("\nBest constant-lambda grid rows:")
print(
    s39_round.groupby("grid_family")
    .head(5)
    .to_string(index=False)
)

print("\nInterpretation helper:")
print(interpretation_df.to_string(index=False))

Mounted at /content/drive
AURORA-TWETF Notebook 18
Exposure-matched constant-lambda, reduced-universe sensitivity, and mechanism figure
RUN_ID: 20260717_082548
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_exposure_matched_lambda_reduced_universe/run_20260717_082548
NOTEBOOK08_INPUT_INDEX: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv

Loading ETF returns and Notebook 07 probability inputs
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
Base ETF return shape: (1426, 4)
20d probability file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_20d_E1_validation_weighted_probability_ensemble.parquet
60d probability file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_mo